In [1]:
import polars as pl

In [ ]:
# Path is relative to the notebook's own folder (run Jupyter from where
# this notebook lives). Update if your data/ folder is elsewhere.
Ipl = pl.read_csv("../data/IPL.csv", ignore_errors=True)
Ipl


## Exploratory Data Analysis

In [3]:
Ipl.group_by("batter").agg(pl.col("runs_batter").sum()).sort(
    "runs_batter", descending=True
)


batter,runs_batter
str,i64
"""V Kohli""",8899
"""RG Sharma""",7185
"""S Dhawan""",6769
"""DA Warner""",6567
"""SK Raina""",5536
…,…
"""YA Abdulla""",0
"""RP Meredith""",0
"""Mayank Dagar""",0


* top batters

In [4]:
Ipl.group_by("batter").agg(pl.col("balls_faced").sum()).sort(
    "balls_faced", descending=True
)

batter,balls_faced
str,i64
"""V Kohli""",6667
"""RG Sharma""",5420
"""S Dhawan""",5326
"""DA Warner""",4702
"""AM Rahane""",4127
…,…
"""Ashok Sharma""",1
"""NJ Rimmington""",1
"""PVSN Raju""",1


* most balls faced

In [5]:
(
    Ipl.group_by("batter")
    .agg(
        [
            pl.col("runs_batter").sum().alias("total_runs"),
            pl.col("balls_faced").sum().alias("total_balls"),
        ]
    )
    .with_columns(
        (pl.col("total_runs") / pl.col("total_balls") * 100).alias("strike_rate")
    )
    .filter(pl.col("total_balls") >= 1000)
    .sort("strike_rate", descending=True)
)


batter,total_runs,total_balls,strike_rate
str,i64,i64,f64
"""AD Russell""",2655,1525,174.098361
"""SP Narine""",1816,1090,166.605505
"""Abhishek Sharma""",1945,1174,165.672913
"""H Klaasen""",1704,1029,165.597668
"""N Pooran""",2335,1414,165.134371
…,…,…,…
"""MK Tiwary""",1695,1449,116.977226
"""R Dravid""",2174,1882,115.515409
"""TM Dilshan""",1153,1007,114.49851


*Best strike rate

In [7]:
(
    Ipl.filter(
        (pl.col("runs_batter").is_in([4, 6])) & (pl.col("runs_not_boundary") == False)
    )
    .group_by("batter")
    .agg(
        [
            (pl.col("runs_batter") == 4).sum().alias("fours"),
            (pl.col("runs_batter") == 6).sum().alias("sixes"),
        ]
    )
    .with_columns((pl.col("fours") + pl.col("sixes")).alias("total_boundaries"))
    .sort("total_boundaries", descending=True)
)


batter,fours,sixes,total_boundaries
str,u32,u32,u32
"""V Kohli""",798,299,1097
"""RG Sharma""",653,310,963
"""S Dhawan""",768,152,920
"""DA Warner""",663,236,899
"""CH Gayle""",407,359,766
…,…,…,…
"""AN Ghosh""",1,0,1
"""Ankit Soni""",0,1,1
"""Arshad Khan (2)""",1,0,1


* most boundaries

## for bowlers

In [9]:
Ipl.group_by("bowler").agg(pl.col("bowler_wicket").sum()).sort(
    "bowler_wicket", descending=True
)


bowler,bowler_wicket
str,i64
"""YS Chahal""",224
"""B Kumar""",205
"""SP Narine""",195
"""PP Chawla""",192
"""R Ashwin""",187
…,…
"""Gurnoor Brar""",0
"""N Wadhera""",0
"""Mashrafe Mortaza""",0


* top bowlers

In [10]:
Ipl.group_by("bowler").agg(pl.col("runs_bowler").sum()).sort(
    "runs_bowler", descending=True
)

bowler,runs_bowler
str,i64
"""R Ashwin""",5652
"""B Kumar""",5586
"""RA Jadeja""",5245
"""YS Chahal""",5159
"""PP Chawla""",5108
…,…
"""AM Rahane""",5
"""Ishan Kishan""",4
"""Atharva Taide""",4


*most balls bowled

In [12]:
(
    Ipl.group_by("bowler")
    .agg(
        [
            pl.col("bowler_wicket").sum().alias("total_wickets"),
            pl.col("runs_bowler").sum().alias("total_runs"),
            pl.col("valid_ball").sum().alias("total_balls"),
        ]
    )
    .with_columns(
        [
            (pl.col("total_runs") / pl.col("total_balls") * 6).alias("economy_rate"),
            (pl.col("total_balls") / pl.col("total_wickets")).alias(
                "bowling_strike_rate"
            ),
        ]
    )
    .filter(pl.col("total_balls") >= 300)
    .sort("total_wickets", descending=True)
)


bowler,total_wickets,total_runs,total_balls,economy_rate,bowling_strike_rate
str,i64,i64,i64,f64,f64
"""YS Chahal""",224,5159,3869,8.000517,17.272321
"""B Kumar""",205,5586,4342,7.719023,21.180488
"""SP Narine""",195,5036,4441,6.803873,22.774359
"""PP Chawla""",192,5108,3850,7.960519,20.052083
"""R Ashwin""",187,5652,4710,7.2,25.187166
…,…,…,…,…,…
"""Pankaj Singh""",11,468,300,9.36,27.272727
"""DJ Hooda""",10,546,379,8.643799,37.9
"""R Parag""",9,493,307,9.635179,34.111111


## Fielding

In [24]:
# Ipl.filter(pl.col("wicket_kind") == "caught").group_by("fielders").agg(
#     pl.len()
# ).sum().sort("fielders", descending=True)
Ipl.filter(pl.col("wicket_kind") == "caught").group_by("fielders").agg(
    pl.len().alias("catches")
).sort("catches", descending=True)

fielders,catches
str,u32
"""MS Dhoni""",158
"""KD Karthik""",145
"""V Kohli""",120
"""AB de Villiers""",120
"""SK Raina""",106
…,…
"""Sumit Kumar""",1
"""AS Yadav""",1
"""TS Mills""",1


## Match-level dataset (deduplicated)
Ipl is ball-by-ball (one row per delivery). Team win counts and toss-impact analysis need one row per match, not one row per ball — otherwise longer matches get overcounted.

In [ ]:
Matches = Ipl.select(["match_id", "match_won_by", "toss_winner"]).unique(
    subset=["match_id"]
)
Matches


In [ ]:
Matches.group_by("match_won_by").agg(pl.len().alias("wins")).sort(
    "wins", descending=True
)


In [ ]:
Matches.with_columns(
    (pl.col("toss_winner") == pl.col("match_won_by")).alias("toss_win_match_win")
).group_by("toss_win_match_win").agg(pl.len().alias("count"))


In [27]:
Ipl.group_by("season").agg(
    pl.col("runs_total").sum().alias("total_runs"),
    pl.col("match_id").n_unique().alias("total_matches"),
).with_columns(
    (pl.col("total_runs") / pl.col("total_matches")).alias("avg_runs_per_match")
).sort("season")


season,total_runs,total_matches,avg_runs_per_match
str,i64,u32,f64
"""2007/08""",17937,58,309.258621
"""2009""",16353,57,286.894737
"""2009/10""",18883,60,314.716667
"""2011""",21154,73,289.780822
"""2012""",22453,74,303.418919
…,…,…,…
"""2022""",24395,74,329.662162
"""2023""",25688,74,347.135135
"""2024""",25971,71,365.788732


In [28]:
Ipl.filter(
    (pl.col("runs_batter") == 6) & (pl.col("runs_not_boundary") == False)
).group_by("season").agg(pl.len().alias("total_sixes")).sort("season")


season,total_sixes
str,u32
"""2007/08""",622
"""2009""",508
"""2009/10""",587
"""2011""",639
"""2012""",731
…,…
"""2022""",1062
"""2023""",1124
"""2024""",1260


In [ ]:
Ipl.group_by(["season", "batter"]).agg(pl.col("runs_batter").sum().alias("runs")).sort(
    ["season", "runs"], descending=[False, True]
).group_by("season", maintain_order=True).first()
